# Matrix Reconstruction (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [232]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 1: Load the desired dataset (test.pt / all.pt)
Load correlation matrices.

In [233]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 252
STRIDE = 5
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}'
RUN = 'AE_150dim_0001'
FORWARD_DAYS = 252
RESULTS_JSON_PATH = f'models/{DATASET_NAME}/{FORWARD_DAYS}_days_gap/AE/{RUN}/run_results.json'
DATASET = 'train' #'train', 'val', 'test' or 'all'


if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / FILE_NAME / 'dataset' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap'

dataset_dir = base_dir

if DATASET == 'all':
    MATRIX_FILE = dataset_dir / 'all.pt'
elif DATASET == 'test':
    MATRIX_FILE = dataset_dir / 'test.pt'
elif DATASET == 'train':
    MATRIX_FILE = dataset_dir / 'train.pt'
elif DATASET == 'val':
    MATRIX_FILE = dataset_dir / 'val.pt'

if not MATRIX_FILE.exists():
    raise FileNotFoundError(
        f"File '{MATRIX_FILE.name}' not found in: {dataset_dir.absolute()}"
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Test Matrix: {MATRIX_FILE.name}')

Environment: Local PC
Dataset selected: data_00_20_w252_s5
Test Matrix: train.pt


In [234]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        indices = payload.get('indices', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), indices, meta


corr, indices, meta = load_corr_payload(MATRIX_FILE)

print(f'Correlation tensor shape: {corr.shape}')
print(f'Indices: {indices[:10]}')

Correlation tensor shape: torch.Size([614, 100, 100])
Indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Step 2: Prepare Matrices (Full Dataset)
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.

In [235]:
all_np = corr.numpy().astype(np.float32)
n_matrices, n_assets, _ = all_np.shape

# 1. Calcolo del numero di features (usando // per forzare il risultato a intero)
n_features = n_assets * (n_assets - 1) // 2

# 2. Otteniamo gli indici del triangolo inferiore ESCLUSA la diagonale (k=-1)
tril_idx = np.tril_indices(n_assets, k=-1)

# 3. Estraiamo simultaneamente i valori per l'intero batch di matrici
# extracted_np avrà shape (n_matrices, n_features)
extracted_np = all_np[:, tril_idx[0], tril_idx[1]]

# 4. Riconvertiamo in tensore PyTorch
x_all = torch.from_numpy(extracted_np)

print(f"{'='*40}")
print(f"Number of matrices   : {n_matrices}")
print(f"Original matrix shape: ({n_assets}, {n_assets})")
print(f"Flattened input size : {n_features}")
print(f"Final tensor shape   : {tuple(x_all.shape)}")
print(f"{'='*40}")

Number of matrices   : 614
Original matrix shape: (100, 100)
Flattened input size : 4950
Final tensor shape   : (614, 4950)


## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [236]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None, dropout_prob: float = 0.02):
        super().__init__()

        # Se hidden_dims è None, creiamo un Linear AE semplice
        if hidden_dims is None:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, latent_dim, bias=False),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, input_dim, bias=False),
            )
        else:
            # Caso Deep Autoencoder con Dropout e attivazioni
            dimensions = [input_dim, *hidden_dims, latent_dim]

            # --- ENCODER ---
            encoder_layers = []
            for i in range(len(dimensions) - 1):
                encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
                if i < len(dimensions) - 2:
                    encoder_layers.append(nn.ReLU())
                    encoder_layers.append(nn.Dropout(dropout_prob))
            self.encoder = nn.Sequential(*encoder_layers)

            # --- DECODER ---
            decoder_dims = dimensions[::-1]
            decoder_layers = []
            for i in range(len(decoder_dims) - 1):
                decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
                if i < len(decoder_dims) - 2:
                    decoder_layers.append(nn.ReLU())
                else:
                    decoder_layers.append(nn.Tanh())
            self.decoder = nn.Sequential(*decoder_layers)

    def architecture_signature(self):
        return [
            [int(layer.in_features), int(layer.out_features)]
            for layer in self.encoder
            if isinstance(layer, nn.Linear)
        ]

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [237]:
def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

OUTPUT_DIR_NAME = 'analysis_outputs'
analysis_dir = results_path.parent / OUTPUT_DIR_NAME
analysis_dir.mkdir(parents=True, exist_ok=True)

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
model_type_raw = str(model_cfg.get('model_type', '')).strip().lower()
if model_type_raw in {'linear', 'linearae', 'linear_ae', 'linear-ae'}:
    model_type = 'linearAE'
elif model_type_raw in {'ae', 'autoencoder', 'auto'}:
    model_type = 'AE'
else:
    model_type = 'AE' if hidden_dims is not None else 'linearAE'

if model_type == 'linearAE':
    hidden_dims = None
else:
    if hidden_dims is None:
        raise ValueError('hidden_dims missing for AE in results JSON')

input_dim = int(model_cfg.get('input_dim', x_all.shape[1]))
if input_dim != x_all.shape[1]:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={x_all.shape[1]}')

dropout_prob = model_cfg.get('dropout', 0.02)
if dropout_prob is None:
    dropout_prob = 0.0
dropout_prob = float(dropout_prob)

model = AutoEncoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dims=hidden_dims,
    dropout_prob=dropout_prob,
).to(device)

weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linearAE':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')

Model type: AE | latent_dim=150 | input_dim=4950
Loaded weights: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\252_days_gap\AE\AE_150dim_0001\best_model.pt


In [238]:
print(model)

AutoEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=4950, out_features=2048, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=2048, out_features=1024, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=1024, out_features=150, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=150, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=4950, bias=True)
    (5): Tanh()
  )
)


## Step 5: Latent Space Analysis + Reconstruction Errors Analysis
Encode the matrices into the latent space and analyze feature distributions + Analyse MSE, MAE and Frobenius.

In [239]:
# 1. Funzione di ricostruzione con gestione automatica del device
def compute_reconstruction(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 64, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []
    reconstructions = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())
            recon = model.decoder(z)
            reconstructions.append(recon.cpu().numpy())

    return np.concatenate(latents, axis=0), np.concatenate(reconstructions, axis=0)

# 2. Funzione per gli errori (rimane invariata, ora riceverà matrici quadrate corrette)
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Input shapes mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed

    if original.ndim == 3:
        mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
        mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
        fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

        summary = pd.DataFrame({
            'MSE': mse_per_matrix,
            'MAE': mae_per_matrix,
            'Frobenius': fro_per_matrix,
        })
    elif original.ndim == 2:
        mse_per_vec = np.mean(diff ** 2, axis=1)
        mae_per_vec = np.mean(np.abs(diff), axis=1)
        l2_per_vec = np.linalg.norm(diff, axis=1)

        summary = pd.DataFrame({
            'MSE': mse_per_vec,
            'MAE': mae_per_vec,
            'L2': l2_per_vec,
        })
    else:
        raise ValueError(f'Expected 2D or 3D arrays, got ndim={original.ndim}')

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [240]:
# ==========================================
# ESECUZIONE DEL CALCOLO E RICOSTRUZIONE
# ==========================================

# Eseguiamo l'inferenza (recon_flat avrà shape [N, 4950])
latents_all, recon_flat = compute_reconstruction(model, x_all, batch_size=64)

# --- IL FIX FONDAMENTALE: Ricostruzione Geometrica ---
n_matrices, n_assets, _ = all_np.shape

# Creiamo le matrici vuote 100x100
recon_all = np.zeros((n_matrices, n_assets, n_assets), dtype=np.float32)

# Estraiamo indici del triangolo inferiore (esclusa la diagonale)
tril_idx = np.tril_indices(n_assets, k=-1)

# Applichiamo la simmetria
recon_all[:, tril_idx[0], tril_idx[1]] = recon_flat
recon_all[:, tril_idx[1], tril_idx[0]] = recon_flat

# Riempiamo la diagonale con 1.0 perfetti
diag_idx = np.arange(n_assets)
recon_all[:, diag_idx, diag_idx] = 1.0
# -----------------------------------------------------

# Ora recon_all ha shape [N, 100, 100], calcoliamo gli errori!
errors_df, summary_df = reconstruction_errors(all_np, recon_all)

# ==========================================
# GESTIONE DATAFRAME E SALVATAGGI
# ==========================================

latent_dim = latents_all.shape[1]
latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
latent_df = pd.DataFrame(latents_all, columns=latent_cols)

if len(indices) != len(errors_df):
    raise ValueError(f'Error rows ({len(errors_df)}) do not match indices ({len(indices)})')
errors_df.insert(0, 'matrix_idx', indices)

print(f'Reconstruction error summary ({DATASET} data):')
display(summary_df)

errors_json_path = analysis_dir / f'reconstruction_errors_{DATASET}_{RUN}.json'
summary_json_path = analysis_dir / f'reconstruction_summary_{DATASET}_{RUN}.json'
errors_df.to_json(errors_json_path, orient='records', indent=2)
summary_df.to_json(summary_json_path, orient='records', indent=2)

print(f'Saved per-matrix errors: {errors_json_path}')
print(f'Saved summary stats: {summary_json_path}')

# Load original payload and create extended version with reconstructed matrices
original_payload = torch.load(MATRIX_FILE, map_location='cpu')
if isinstance(original_payload, dict):
    extended_payload = original_payload.copy()
else:
    # If it's just a tensor, wrap it
    extended_payload = {'corr_tensor': original_payload}

# Add reconstructed matrices as a new key
recon_tensor = torch.from_numpy(recon_all).float()
extended_payload['corr_tensor_reconstructed'] = recon_tensor
extended_payload['tickers'] = meta['meta']['tickers']

# Save to file with the same name but in original location
output_path = analysis_dir / f'{DATASET}_reconstructed_{RUN}.pt'
torch.save(extended_payload, output_path)
print(f'Saved reconstructed matrices: {output_path}')

print(f'\nLatent distribution summary ({DATASET} data):')
display(latent_df.describe().T)

valid_cols = [col for col in latent_cols if latent_df[col].notna().any() and latent_df[col].nunique() > 1]
latent_df = latent_df[valid_cols]

if len(indices) != len(latent_df):
    raise ValueError(f'Latent rows ({len(latent_df)}) do not match indices ({len(indices)})')
latent_df.insert(0, 'matrix_idx', indices)

latent_json_path = analysis_dir / f'latent_{DATASET}_{RUN}.json'
latent_df.to_json(latent_json_path, orient='records', indent=2)
print(f'Saved latent samples: {latent_json_path}')

if len(valid_cols) < 2:
    print('Not enough valid latent dimensions for pairwise plots.')
elif len(valid_cols) > 20:
    # Aggiunto il blocco di sicurezza per evitare esplosioni di RAM/tempo
    print(f'Skipping pairwise plot: too many latent dimensions ({len(valid_cols)} > 20).')
else:
    plot_df = latent_df[valid_cols]
    grid = sns.PairGrid(plot_df, corner=True, diag_sharey=False)
    grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
    grid.figure.suptitle(f'Pairwise latent dimension plots ({DATASET} data)', y=1.02)
    pairplot_path = analysis_dir / f'latent_pairwise_{DATASET}_{RUN}.png'
    grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved latent pairwise plot: {pairplot_path}')

Reconstruction error summary (train data):


,mean,std,min,median,max
MSE,0.000421,0.000175,0.000131,0.000385,0.001151
MAE,0.015761,0.003315,0.008859,0.015406,0.025991
Frobenius,2.011298,0.411684,1.143265,1.962541,3.392059


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\252_days_gap\AE\AE_150dim_0001\analysis_outputs\reconstruction_errors_train_AE_150dim_0001.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\252_days_gap\AE\AE_150dim_0001\analysis_outputs\reconstruction_summary_train_AE_150dim_0001.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\252_days_gap\AE\AE_150dim_0001\analysis_outputs\train_reconstructed_AE_150dim_0001.pt

Latent distribution summary (train data):


,count,mean,std,min,25%,50%,75%,max
z1,614.0,-0.006396,0.023030,-0.045188,-0.024522,-0.013589,0.015892,0.036354
z2,614.0,-0.002139,0.006558,-0.017469,-0.005966,-0.003406,0.002486,0.012220
z3,614.0,-0.012452,0.012609,-0.044250,-0.018893,-0.012337,-0.003933,0.011535
z4,614.0,-0.216534,1.554442,-2.797400,-1.452230,-0.393348,1.124302,2.824339
z5,614.0,-1.608169,2.506453,-6.370063,-3.789000,-1.115252,0.209901,3.237980
...,...,...,...,...,...,...,...,...
z146,614.0,0.000606,0.003587,-0.008134,-0.001679,0.001453,0.002977,0.008955
z147,614.0,0.000500,0.001472,-0.002010,-0.000644,0.000146,0.001727,0.003843
z148,614.0,-0.002615,0.012464,-0.022861,-0.012231,-0.006306,0.008985,0.018852
z149,614.0,-0.002572,0.009066,-0.028270,-0.008996,-0.001230,0.004078,0.015785


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\252_days_gap\AE\AE_150dim_0001\analysis_outputs\latent_train_AE_150dim_0001.json
Skipping pairwise plot: too many latent dimensions (150 > 20).
